# Imports

In [3]:
import pandas as pd 
from sklearn.model_selection import train_test_split 

# Load dataset

In [2]:
# load dataset into dataframe
df = pd.read_csv('../../data/data.csv')

In [3]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [4]:
# df.to_parquet('my_cleaned_dataset.parquet')
# df = pd.read_parquet('my_cleaned_dataset.parquet')

In [5]:
# df.head()
# %pip install pyarrow

# Fraud transactions distribution

In [6]:
# 2. Calculate each row's position as a percentage of the total dataset (0% to 100%)
df['row_percentage'] = (df.index / len(df)) * 100

# 3. Create 10 bins (0-10%, 10-20%, etc.) to group the timeline
bins = range(0, 101, 10)
labels = [f"{i}% to {i+10}%" for i in range(0, 100, 10)]
df['timeline_chunk'] = pd.cut(df['row_percentage'], bins=bins, labels=labels, include_lowest=True)

# 4. Group the data by these chunks and count the fraud
# Assuming your target column is named 'isFraud' (where 1 is fraud, 0 is normal)
distribution = df.groupby('timeline_chunk', observed=False)['isFraud'].agg(
    total_transactions='count',
    fraud_cases='sum'
)

# 5. Calculate what percentage of your TOTAL fraud lives in each chunk
total_fraud_count = df['isFraud'].sum()
distribution['%_of_total_frauds'] = (distribution['fraud_cases'] / total_fraud_count) * 100

# Format it nicely for reading
distribution['%_of_total_frauds'] = distribution['%_of_total_frauds'].round(2).astype(str) + '%'

print("--- FRAUD DISTRIBUTION ACROSS TIMELINE ---")
print(distribution)

--- FRAUD DISTRIBUTION ACROSS TIMELINE ---
                total_transactions  fraud_cases %_of_total_frauds
timeline_chunk                                                   
0% to 10%                   636263          383             4.66%
10% to 20%                  636262         1157            14.09%
20% to 30%                  636262          352             4.29%
30% to 40%                  636262          406             4.94%
40% to 50%                  636262          401             4.88%
50% to 60%                  636262          490             5.97%
60% to 70%                  636262          454             5.53%
70% to 80%                  636262          316             3.85%
80% to 90%                  636262          490             5.97%
90% to 100%                 636261         3764            45.83%


In [7]:
df = df.drop(['row_percentage', 'timeline_chunk'], axis=1)
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [3]:
# drop useless columns 
df = df.drop(['oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFlaggedFraud'], axis=1)

In [4]:
df_test = pd.get_dummies(df, columns = ['type'], dtype = int)
del df

In [5]:
df_test.head()

,step,amount,nameOrig,nameDest,isFraud,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,1,9839.64,C1231006815,M1979787155,0,0,0,0,1,0
1,1,1864.28,C1666544295,M2044282225,0,0,0,0,1,0
2,1,181.00,C1305486145,C553264065,1,0,0,0,0,1
3,1,181.00,C840083671,C38997010,1,0,1,0,0,0
4,1,11668.14,C2048537720,M1230701703,0,0,0,0,1,0


In [11]:
df_test.to_parquet('../../data/processed_data.parquet')

In [4]:
df_test = pd.read_parquet('../../data/processed_data.parquet')

In [5]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 10 columns):
 #   Column         Dtype  
---  ------         -----  
 0   step           int64  
 1   amount         float64
 2   nameOrig       object 
 3   nameDest       object 
 4   isFraud        int64  
 5   type_CASH_IN   int64  
 6   type_CASH_OUT  int64  
 7   type_DEBIT     int64  
 8   type_PAYMENT   int64  
 9   type_TRANSFER  int64  
dtypes: float64(1), int64(7), object(2)
memory usage: 485.4+ MB


In [6]:
# ORIGINATOR PERSPECTIVE: sort by sender + time 
df_test = df_test.sort_values(['nameOrig', 'step']).reset_index(drop=True)

# Transaction Velocity (how many tx in same step/hour)
df_test['orig_tx_count_same_step'] = df_test.groupby(['nameOrig', 'step'])['step'].transform('count')
# Cumulative tx count per person up to this point
df_test['orig_tx_cumcount']        = df_test.groupby('nameOrig').cumcount() + 1
# Escalating amount pattern (current vs previous tx amount)
df_test['orig_prev_amount']        = df_test.groupby('nameOrig')['amount'].shift(1)
df_test['orig_amount_ratio']       = df_test['amount'] / (df_test['orig_prev_amount'] + 1)
# Amount deviation from that person's own average

# Has this destination been hit multiple times? (mule account signal)
df_test['orig_prev_step']          = df_test.groupby('nameOrig')['step'].shift(1)
df_test['steps_since_last_tx']     = df_test['step'] - df_test['orig_prev_step']

'\nnow everything is in order of \n\n- nameOrig \n   - step \n   \norig_tx_count_same_step = \n\n'

In [7]:
# DESTINATION PERSPECTIVE: re-sort by receiver + time
df_test = df_test.sort_values(['nameDest', 'step']).reset_index(drop=True)

# 1. The Burst Counter (How many times received in this exact hour)
# df_test['dest_tx_count_same_step'] = df_test.groupby(['nameDest', 'step'])['step'].transform('count')
# 2. The Lifetime Counter (How many times received TOTAL)
df_test['dest_tx_cumcount']        = df_test.groupby('nameDest').cumcount() + 1
df_test['dest_prev_amount']        = df_test.groupby('nameDest')['amount'].shift(1)
df_test['dest_amount_ratio']       = df_test['amount'] / (df_test['dest_prev_amount'] + 1)  # escalation into dest
df_test['dest_prev_step']          = df_test.groupby('nameDest')['step'].shift(1)
df_test['dest_steps_since_last']   = df_test['step'] - df_test['dest_prev_step']  # how frequently is dest receiving?

In [8]:
# PAIR PERSPECTIVE: orig-dest relationship (no sort needed, groupby handles it)
# to know how many times a user transfers to the same dest 
df_test['pair_tx_cumcount']        = df_test.groupby(['nameOrig', 'nameDest']).cumcount()
# Is this a new relationship? (first time A sends to B)
df_test['is_new_dest']             = (df_test['pair_tx_cumcount'] == 0).astype(int)
df_test['pair_total_amount'] = df_test.groupby(['nameOrig', 'nameDest'])['amount'].cumsum()

In [1]:
df_test = df_test.sort_values('step').reset_index(drop=True)

NameError: name 'df_test' is not defined

In [ ]:
# Fill the missing historical amounts and ratios with 0
df_test.fillna({
    'orig_prev_amount': 0,
    'orig_amount_ratio': 0,
    'dest_prev_amount': 0,
    'dest_amount_ratio': 0,
}, inplace=True)

# Fill the missing time steps with -1 (Explicitly meaning "First Time Transaction")
df_test.fillna({
    'steps_since_last_tx': -1,
    'dest_steps_since_last': -1
}, inplace=True)

In [ ]:
# Time of day (step cycles every 24 hours)
df_test['hour_of_day'] = df_test['step'] % 24          # 0–23, fraud may spike at odd hours

# Day number 
df_test['day'] = df_test['step'] // 24                 # which day of the simulation

# ── 3. Is it a late night transaction? (high fraud risk window) ───────────────
df_test['is_night'] = ((df_test['hour_of_day'] < 6) | (df_test['hour_of_day'] >= 22)).astype(int)

In [ ]:
df_test = df_test.drop(columns=[
    # 'nameOrig', 
    'nameDest',
    'orig_prev_step', 
    'dest_prev_step',
    'step'
]
                      )

In [ ]:
df_test.info()

In [ ]:
df_test.isnull().sum()

In [ ]:
len(df_test)

In [16]:
# df_test.to_parquet('../../data/processed_grouped_data.parquet')
df_test.to_parquet('../../data/grouped_data_with_name.parquet')